# Bounded agent loops with Claude

How do you keep a multi-step agent loop from spiralling? A model plans some tool calls, executes them, reflects on the results, decides whether to keep going, and if so plans the next batch. Without explicit guards that loop is unbounded. A bad first plan, an over-eager reflector, or just a hard query can keep it iterating long past the point of useful work. The mean cost per query stays manageable; the worst case wrecks the unit economics.

The fix is a small kit of guards, each enforcing a different kind of termination:

- An **iteration cap**: a hard ceiling regardless of what the reflector says.
- A **per-request token budget**: a soft ceiling that bails when projected cost exceeds a threshold.
- A **structured reflector**: a principled early exit where the model commits to a should_continue boolean via a Pydantic schema, rather than narrating its way around the decision.

The first two bound the worst-case spend. The third makes termination an explicit, auditable decision rather than something implicit in the planner behaviour. Long-tail cost distributions are the symptom most teams notice first; reliable termination is what fixes them.

This guide walks through all three on a runnable agent that answers multi-hop geography questions using two free APIs (REST Countries, Wikipedia) and Claude. At the end there is an evaluation harness measuring iteration count, cost, and answer correctness across a labelled set, comparing a naive loop to a guarded one.

## Where this comes from

This pattern is drawn from a personal project: a LangGraph-based transfer-recommendation agent for Fantasy Premier League, running on AWS Lambda behind CloudFront. The unguarded version's cost distribution had a long tail that I bounded by adding these three guards after the first month of production runs. The cookbook version below is a portability rewrite: no AWS dependencies, mock tools via free public APIs, plain Python instead of LangGraph. The techniques and the failure modes come from production; the demo is just easier to run.

## Setup

In [1]:
%%capture
%pip install anthropic requests python-dotenv pydantic

In [2]:
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal
from urllib.parse import quote

import anthropic
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()
# max_retries bumped from the default of 2 to ride out transient overload windows
# (HTTP 529) gracefully on a long-running notebook execution.
client = anthropic.Anthropic(max_retries=5)

We use two models. Haiku handles the high-volume work where speed and cost dominate: planning, tool argument construction, summarising tool outputs. Sonnet handles the reflector and recommender, where the model is making judgement calls that affect whether the loop continues or terminates. This split is the first cost lever. In a typical run the reflector is called once per iteration and the executor is called multiple times per iteration, so spending the cheap model on the executor is what keeps total cost manageable.

In [3]:
HAIKU = "claude-haiku-4-5"
SONNET = "claude-sonnet-4-6"

## A four-node agent loop

The agent is a state machine with four nodes. State is a plain dataclass that gets
threaded through every node and accumulates the trace.

- `planner`: turns the query (or the previous iteration's reflection) into a list of
  tool calls to run next.
- `executor`: runs the planned tool calls and appends their outputs to the trace.
- `reflector`: reads the trace and decides whether the answer is in reach or another
  plan is needed.
- `recommender`: writes the final answer from the accumulated trace.

The edge from `reflector` back to `planner` is the only conditional edge. Everything
else is linear.

In [4]:
@dataclass
class AgentState:
    query: str
    plan: list[dict[str, Any]] = field(default_factory=list)
    trace: list[dict[str, Any]] = field(default_factory=list)
    reflection: str = ""
    answer: str = ""
    iterations: int = 0
    tokens_in: int = 0
    tokens_out: int = 0
    should_continue: bool = True  # set by reflector; structured reflector toggles this

### Tools

Two real, free, no-auth APIs. `country_info` returns structured facts (population,
capital, languages) from the REST Countries service. `wikipedia_summary` returns the
first paragraph of a Wikipedia article. The multi-hop queries we'll feed the agent
force it to combine the two.

In [5]:
WIKI_API = "https://en.wikipedia.org/api/rest_v1/page/summary/"
COUNTRIES_API = "https://restcountries.com/v3.1/name/"
HEADERS = {"User-Agent": "claude-cookbooks/1.0 (cost-aware-agent-loops)"}


def country_info(name: str) -> dict[str, Any]:
    r = requests.get(COUNTRIES_API + quote(name), headers=HEADERS, timeout=10)
    if r.status_code == 404:
        return {"error": f"no country matched '{name}'"}
    r.raise_for_status()
    hit = r.json()[0]
    return {
        "name": hit["name"]["common"],
        "capital": hit.get("capital", [None])[0],
        "region": hit.get("region"),
        "population": hit.get("population"),
        "languages": list(hit.get("languages", {}).values()),
        "currencies": list(hit.get("currencies", {}).keys()),
    }


def wikipedia_summary(title: str) -> str:
    r = requests.get(WIKI_API + quote(title.replace(" ", "_")), headers=HEADERS, timeout=10)
    if r.status_code == 404:
        return f"no Wikipedia article matched '{title}'"
    r.raise_for_status()
    return r.json()["extract"]


TOOLS = {
    "country_info": country_info,
    "wikipedia_summary": wikipedia_summary,
}


TOOL_SCHEMA = """Available tools:

- country_info(name: str) -> dict
    Returns structured facts (capital, region, population, official languages,
    currencies) for a country. Use exact country names.

- wikipedia_summary(title: str) -> str
    Returns the first paragraph of a Wikipedia article. Use for facts that aren't
    in the country_info structured fields (e.g. capital city populations, historical
    detail)."""

### The four nodes

Each node is a plain function from state to state. The planner and reflector use
structured outputs so we can rely on the fields by name; the executor is deterministic
Python; the recommender returns free text.

In [6]:
class ToolCall(BaseModel):
    tool: Literal["country_info", "wikipedia_summary"]
    argument: str = Field(description="The argument to pass to the tool")
    rationale: str = Field(description="One short sentence on why this call helps answer the query")


class Plan(BaseModel):
    calls: list[ToolCall] = Field(description="Tool calls to run this iteration (1-3 is typical)")


def planner(state: AgentState) -> AgentState:
    """Turn the query plus accumulated trace into the next batch of tool calls.

    The "do not skip tools because you think you know the answer" clause in the
    prompt is a demonstration crutch, not production guidance. Haiku has enough
    world knowledge to answer most of the geography queries below in zero tool
    calls; forcing it to use tools is how we exercise the loop mechanics that
    this notebook is teaching. In a real agent, you'd usually want the planner
    to short-circuit when it can, and the cost-control techniques here apply
    only to queries that genuinely need iteration.
    """
    context = (
        f"User query: {state.query}\n\n"
        f"{TOOL_SCHEMA}\n\n"
        f"Previous tool outputs:\n{json.dumps(state.trace, indent=2) if state.trace else '(none yet)'}\n\n"
        f"Reflection from previous iteration:\n{state.reflection or '(none)'}\n\n"
        f"Plan the next 1-3 tool calls. Return an empty list ONLY when every fact "
        f"needed to answer the query is already present in the tool outputs above. "
        f"Do not skip tools because you think you know the answer from prior "
        f"knowledge. The user wants tool-grounded facts."
    )
    response = client.messages.parse(
        model=HAIKU,
        max_tokens=800,
        messages=[{"role": "user", "content": context}],
        output_format=Plan,
    )
    state.tokens_in += response.usage.input_tokens
    state.tokens_out += response.usage.output_tokens
    state.plan = [c.model_dump() for c in response.parsed_output.calls]
    return state


def executor(state: AgentState) -> AgentState:
    """Run the planned tool calls. No LLM calls here; deterministic Python."""
    for call in state.plan:
        try:
            output = TOOLS[call["tool"]](call["argument"])
        except (requests.RequestException, KeyError) as e:
            output = {"error": str(e)}
        state.trace.append({"call": call, "output": output})
    state.plan = []
    return state

Below is the reflector. The naive version returns a free-text reflection that
shapes the next iteration's planning context, but with no explicit termination
signal of its own. The naive loop trusts the planner to know when it's done by
returning an empty plan. The guarded version (we'll add later) returns a
structured object whose `should_continue` boolean is an explicit termination
signal independent of the planner.

In [7]:
def reflector_naive(state: AgentState) -> AgentState:
    """Free-text reflection. The loop reads it and decides whether to continue based
    on heuristics (e.g. did the model say 'I have enough')."""
    context = (
        f"User query: {state.query}\n\n"
        f"Tool outputs so far:\n{json.dumps(state.trace, indent=2)}\n\n"
        f"Reflect on whether the answer is in reach. If yes, say so explicitly. "
        f"If not, describe what's missing."
    )
    response = client.messages.create(
        model=SONNET,
        max_tokens=400,
        messages=[{"role": "user", "content": context}],
    )
    state.tokens_in += response.usage.input_tokens
    state.tokens_out += response.usage.output_tokens
    state.reflection = response.content[0].text
    return state


def recommender(state: AgentState) -> AgentState:
    """Synthesise the final answer from the accumulated trace."""
    context = (
        f"User query: {state.query}\n\n"
        f"All tool outputs collected:\n{json.dumps(state.trace, indent=2)}\n\n"
        f"Answer the query in one or two sentences. If the trace doesn't contain "
        f"enough information, say so explicitly rather than guessing."
    )
    response = client.messages.create(
        model=SONNET,
        max_tokens=400,
        messages=[{"role": "user", "content": context}],
    )
    state.tokens_in += response.usage.input_tokens
    state.tokens_out += response.usage.output_tokens
    state.answer = response.content[0].text
    return state

## Running the loop with no guards

This is the loop most agent tutorials show. The reflector is called for its
free-text reflection, which feeds the next iteration's planner context, but
the loop itself has no explicit termination signal from it. Termination
happens when the planner returns an empty plan (or the absolute safety net at
10 iterations binds, which is itself a kind of guard but a very coarse one).

A note on the baseline: the unguarded loop here still calls a Sonnet reflector
every iteration. A strictly cheaper "no guards" pattern would skip the
reflector entirely. The reflector is kept here for parity with the guarded
loop below, so the only thing varying between configurations is the
termination logic (and the iteration / budget caps), not the per-iteration
work. If you want to size the cost of the reflector itself in isolation, run
both with and without it on your workload.

In [8]:
def run_unguarded(query: str) -> AgentState:
    """The naive pattern: trust the planner to return an empty plan when it's done.

    The reflector is called for its content (it shapes the next iteration's planning
    context) but the loop has no explicit termination signal of its own. This is a
    realistic pattern in agent tutorials; its weakness is that the planner has to be
    both competent at planning AND honest about completion, which are different skills.
    """
    state = AgentState(query=query)
    while True:
        state.iterations += 1
        state = planner(state)
        if not state.plan:  # planner self-terminates by returning no calls
            break
        state = executor(state)
        state = reflector_naive(state)
        # safety net so a runaway demo doesn't bankrupt the reader
        if state.iterations > 10:
            break
    state = recommender(state)
    return state

Try it on a single multi-hop query. Note the iteration count and the token totals.

In [9]:
demo_query = (
    "Among the G7 countries (United States, United Kingdom, Canada, France, "
    "Germany, Italy, Japan), which has the largest population?"
)
result = run_unguarded(demo_query)
print(f"Answer: {result.answer}")
print(f"Iterations: {result.iterations}")
print(f"Tokens in / out: {result.tokens_in} / {result.tokens_out}")

Answer: Based on the data retrieved, the **United States** has the largest population among the G7 countries, with approximately **340 million people**. Here's the full ranking for context: United States (340.1M) > Japan (123.2M) > Germany (83.5M) > United Kingdom (69.3M) > France (66.4M) > Italy (58.9M) > Canada (41.7M).
Iterations: 2
Tokens in / out: 4263 / 577


The query above needs information on seven countries, which is more than the
planner can bundle into a single iteration of 1-3 tool calls. So the loop runs
multiple iterations: plan a batch, execute, reflect, plan the next batch, repeat.
On a clean run this terminates in 3-4 iterations. On a less clean run, same query
but a different planner plan, it can take 6+ iterations. The variance is the
problem. The unit economics of an agent product depend on the cost-per-query
distribution, not the median; a long tail of expensive runs is what kills the
product.

## Iteration cap

The simplest guardrail. Pass a `max_iterations` parameter and stop unconditionally
when it binds, regardless of what the reflector says.

**The right value is empirical.** For most agent loops, 90% of correct answers are
reachable within 3 iterations and the remaining 10% are either unreachable or only
reachable after enough extra work that you'd rather degrade gracefully than pay for
the long tail. We'll measure this in the evaluation section.

In [10]:
def run_with_iteration_cap(query: str, max_iterations: int = 3) -> AgentState:
    state = AgentState(query=query)
    while state.iterations < max_iterations:
        state.iterations += 1
        state = planner(state)
        if not state.plan:
            break
        state = executor(state)
        state = reflector_naive(state)
    state = recommender(state)
    return state

In [11]:
result = run_with_iteration_cap(demo_query, max_iterations=3)
print(f"Answer: {result.answer}")
print(f"Iterations: {result.iterations}  (cap: 3)")
print(f"Tokens: {result.tokens_in + result.tokens_out}")

Answer: Based on the retrieved data, the **United States** has the largest population among the G7 countries, with approximately **340 million people**. For reference, the full G7 ranking by population is: United States (340.1M) > Japan (123.2M) > Germany (83.5M) > United Kingdom (69.3M) > France (66.4M) > Italy (58.9M) > Canada (41.7M).
Iterations: 2  (cap: 3)
Tokens: 5920


## Per-request budget cap

Iteration count is one axis. Tokens-per-iteration is another. A query that pulls in
a long Wikipedia article and then has to summarise the trace can blow through tokens
in a single iteration even with the count cap. The cleaner mental model is "cap the
request, not the iteration".

Anthropic's API returns usage on every response. We track running totals on the state
and check before each LLM call whether the next call would exceed the budget. If yes,
the loop terminates and the recommender runs on what we have.

In [12]:
@dataclass
class Budget:
    max_tokens_in: int = 30_000
    max_tokens_out: int = 5_000
    # Rough pre-check estimate so we don't make the call that breaks the budget.
    # Tune to typical call sizes on your workload.
    estimated_call_in: int = 3_000
    estimated_call_out: int = 500


def would_exceed_budget(state: AgentState, budget: Budget) -> bool:
    projected_in = state.tokens_in + budget.estimated_call_in
    projected_out = state.tokens_out + budget.estimated_call_out
    return projected_in > budget.max_tokens_in or projected_out > budget.max_tokens_out

The check is approximate; you are estimating the next call size before making it.
That is fine for a budget guard. The point is not to refuse the last £0.01 call; it is
to stop runaway loops before they reach the breaking point. Tune the estimates from
your own logs once you have run the workload for a week.

Where to store the budget counter matters for multi-process agents. An in-process
dataclass is fine for a synchronous handler. A distributed agent (multiple Lambda
invocations sharing a logical request, say) needs an external store: DynamoDB with
atomic counter updates, or Redis with INCRBY. The pattern is the same; only the
storage changes.

In [13]:
def run_with_budget(
    query: str, max_iterations: int = 3, budget: Budget | None = None
) -> AgentState:
    budget = budget or Budget()
    state = AgentState(query=query)
    while state.iterations < max_iterations:
        if would_exceed_budget(state, budget):
            state.reflection = "(budget cap reached, terminating early)"
            break
        state.iterations += 1
        state = planner(state)
        if not state.plan:
            break
        state = executor(state)
        if would_exceed_budget(state, budget):
            state.reflection = "(budget cap reached after executor, skipping reflector)"
            break
        state = reflector_naive(state)
    state = recommender(state)
    return state

## Structured reflector judgement

This is the technique I'd argue matters most for decision quality, from
production intuition rather than from measured contribution on the eval below.
It is also the one most agent tutorials skip. The naive loop above trusts the
planner to return an empty plan when it has enough, which pushes two different
decisions onto one node: what to call next, AND when to stop. Those are
different skills. A planner that is competent at the first can be unreliable
at the second. It stops early when it half-remembers an answer from training,
or it keeps planning calls forever because the trace has facts but the planner
is uncertain whether they are enough.

The fix is to add an explicit termination signal as its own node, separate from
planning. A structured output with `should_continue: bool` and `confidence:
float` forces the model to commit on the question "do we have enough yet". The
boolean is the loop control signal; the confidence score is for observability
(calibration drift over time tells you something is wrong with the prompts).

The eval at the end of this notebook compares the unguarded loop to the bundle
of all three guards, not against each guard in isolation. If you care which one
is doing the work for your shape of query, run the ablation yourself.

In [14]:
class Reflection(BaseModel):
    summary: str = Field(description="One sentence on what the trace tells us so far")
    should_continue: bool = Field(
        description="True only if another iteration is likely to materially improve "
        "the answer. False if the trace is sufficient OR if more iterations won't help."
    )
    confidence: float = Field(
        ge=0.0, le=1.0, description="0.0 = no idea, 1.0 = certain. Calibrate honestly."
    )
    what_is_missing: str = Field(
        description="If should_continue is True, what specifically is needed. "
        "If False, leave empty."
    )


def reflector_structured(state: AgentState) -> AgentState:
    context = (
        f"User query: {state.query}\n\n"
        f"Tool outputs so far:\n{json.dumps(state.trace, indent=2)}\n\n"
        f"Decide whether another iteration would help. Be honest about diminishing "
        f"returns: if the trace already contains the answer, set should_continue=False "
        f"even if you could imagine more calls. If the query is fundamentally "
        f"unanswerable with the available tools, set should_continue=False."
    )
    response = client.messages.parse(
        model=SONNET,
        max_tokens=400,
        messages=[{"role": "user", "content": context}],
        output_format=Reflection,
    )
    state.tokens_in += response.usage.input_tokens
    state.tokens_out += response.usage.output_tokens
    parsed = response.parsed_output
    state.reflection = (
        f"{parsed.summary} (continue={parsed.should_continue}, confidence={parsed.confidence:.2f})"
    )
    state.should_continue = parsed.should_continue
    return state

## Putting it all together

The guarded loop combines all three: iteration cap, budget cap, structured reflector.
The control flow is cleaner than the naive version because the termination signal is
explicit rather than parsed from prose.

In [15]:
def run_guarded(query: str, max_iterations: int = 3, budget: Budget | None = None) -> AgentState:
    budget = budget or Budget()
    state = AgentState(query=query)
    while state.iterations < max_iterations:
        if would_exceed_budget(state, budget):
            state.reflection = "(budget cap reached, terminating early)"
            break
        state.iterations += 1
        state = planner(state)
        if not state.plan:
            break
        state = executor(state)
        if would_exceed_budget(state, budget):
            state.reflection = "(budget cap reached after executor, skipping reflector)"
            break
        state = reflector_structured(state)
        if not state.should_continue:
            break
    state = recommender(state)
    return state

In [16]:
result = run_guarded(demo_query, max_iterations=3)
print(f"Answer: {result.answer}")
print(f"Iterations: {result.iterations}")
print(f"Tokens: {result.tokens_in + result.tokens_out}")
print(f"Final reflection: {result.reflection}")

Answer: Based on the data retrieved, the **United States** has the largest population among G7 countries, with approximately **340 million people** — nearly three times larger than the second-most populous G7 member, Japan (~123 million).
Iterations: 1
Tokens: 3261
Final reflection: All 7 G7 countries have been queried and their populations retrieved, making it possible to directly compare and identify the largest. (continue=False, confidence=0.99)


## Evaluation

Three measurements matter for an agent loop. **Cost** (tokens per query, dollars per
query); **correctness** (does the final answer match the gold answer); **iteration
count** (is the cap binding, and does binding hurt correctness). The interesting
question is the trade-off: do the guards reduce cost without losing accuracy, or
does each guard cost some recall?

We ship a small hand-labelled set in `data/sample_queries.json` (six multi-hop
geography queries with unambiguous answers) and a scorer in
`evaluation/eval_loop_efficiency.py`. The scorer runs the unguarded and guarded loops
over the set and reports the trade-off as a table.

In [17]:
DATA_DIR = Path("data")
with open(DATA_DIR / "sample_queries.json", encoding="utf-8") as f:
    eval_set = json.load(f)
print(f"Loaded {len(eval_set)} labelled queries.")
print(f"First: {eval_set[0]['query']}")
print(f"Gold:  {eval_set[0]['gold_contains']}")

Loaded 8 labelled queries.
First: Which Nordic country has Icelandic as its official language?
Gold:  ['Iceland']


A minimal scorer checks whether the gold phrase appears (case-insensitive) in the
answer string. This is intentionally simple: the queries are designed so the right
answer is a specific noun phrase. For more nuanced evaluation, swap in an LLM-judge
pattern.

In [18]:
def score_answer(answer: str, gold_contains: list[str]) -> bool:
    return all(phrase.lower() in answer.lower() for phrase in gold_contains)


def evaluate(run_fn, eval_set: list[dict], label: str) -> dict:
    results = []
    for case in eval_set:
        state = run_fn(case["query"])
        correct = score_answer(state.answer, case["gold_contains"])
        results.append(
            {
                "query": case["query"],
                "correct": correct,
                "iterations": state.iterations,
                "tokens": state.tokens_in + state.tokens_out,
            }
        )
    n = len(results)
    return {
        "label": label,
        "accuracy": sum(r["correct"] for r in results) / n,
        "mean_iterations": sum(r["iterations"] for r in results) / n,
        "mean_tokens": sum(r["tokens"] for r in results) / n,
        "max_tokens": max(r["tokens"] for r in results),
        "per_query": results,
    }

In [19]:
# Comment out one of the two below if you only want to evaluate one configuration.
unguarded_results = evaluate(run_unguarded, eval_set, "unguarded")
guarded_results = evaluate(
    run_guarded, eval_set, "guarded (cap=3, budget=30k/5k, structured reflector)"
)

print(f"{'config':<60} {'acc':>5} {'iter':>5} {'mean_tok':>9} {'max_tok':>8}")
for r in (unguarded_results, guarded_results):
    print(
        f"{r['label']:<60} {r['accuracy']:>5.2f} {r['mean_iterations']:>5.1f} "
        f"{r['mean_tokens']:>9.0f} {r['max_tokens']:>8.0f}"
    )

config                                                         acc  iter  mean_tok  max_tok
unguarded                                                     1.00   4.9     20769    68829
guarded (cap=3, budget=30k/5k, structured reflector)          1.00   2.0      5570    10069


Reading the numbers. On a typical run of this set, the guarded loop drops mean tokens with no accuracy loss, and the max_tokens column tightens more sharply than the mean: that is the cap-bounded tail, which is what matters for unit economics.

What the trade-off costs, when it bites. On the standalone scorer run shipped alongside this notebook, one query (which of the five most populous European countries has the smallest land area) lost accuracy under the guards. The unguarded loop reached iteration 8 on that query and got it right. The guarded run terminated at iteration 1 with the structured reflector saying should_continue=False, and the recommender answered from the partial trace and got it wrong. This is what guards cost when they bind on a query that needed more rope.

Treat this as illustrative, not a benchmark. Eight queries shows the shape of the trade-off; run your own eval across more queries on your workload if you want a number to point at.

Two structural caveats on the comparison:

1. **The comparison is bundle versus none, not ablated.** The guarded loop combines the iteration cap, the budget cap, and the structured reflector. The eval does not isolate which guard is doing how much of the work. On this query set, either the cap or the structured reflector alone would catch most of the cost; for your workload, you would want to ablate to see which one matters most.

2. **Run-to-run variance is real.** Both the planner and the reflector are non-deterministic, and the mean-token reduction varies meaningfully between runs of the same code.

If your accuracy drops measurably under the guards, two things to check first: (a) is the iteration cap too tight for this workload? Try 4 or 5. (b) Is the structured reflector setting should_continue=False too aggressively? Add a few examples of "yes you should continue" cases to the prompt as few-shot.

## Scaling up

This notebook runs a synchronous loop on a single process with an in-memory budget
counter. Three things change as you move to production:

**Budget counter goes external.** For a single Lambda invocation per query, the
in-process counter is fine. For an agent that fans out across multiple processes
(parallel sub-agents, distributed tool calls), you need an atomic shared counter:
DynamoDB with conditional updates is cheap and survives cold starts, Redis INCRBY is
faster if you already have Redis. The pattern is identical to in-process; only the
storage moves.

**Observability needs per-node tracing.** The eval harness here computes aggregate
stats. In production you want per-call traces (which iteration, which tool, which
model, what the inputs and outputs looked like) so you can answer questions like
"why did this run cost 3x the median". Langfuse, Phoenix, or any OpenTelemetry-shaped
tracer works; the structured-reflector boolean and confidence are the high-value
fields to log.

**Idempotency on retries.** If the loop terminates mid-iteration (Lambda timeout,
upstream failure) and the user retries, you do not want to pay for the work twice.
Persist the state by a request ID after each node completes; on retry, resume from
the last completed node rather than restarting from scratch. This requires the nodes
to be pure (no side effects beyond the state mutation), which is why the design above
threads state through every node rather than holding closures over local variables.

The three techniques here (iteration caps, budget caps, structured reflector) are
the agent-loop equivalent of "always set a timeout on your HTTP requests". They are
obvious in retrospect, omitted by default in most tutorials, and the difference
between a workload that works and a workload that is quietly haemorrhaging money.